# OOP OpenAI Integration with MCP - Step-by-Step

This notebook breaks down the `client.py` script into executable cells. It connects to the `server.py` MCP server using an Object-Oriented approach (`MCPOpenAIClient`).

We will build the class method by method and attach them dynamically. We'll also add plenty of `print()` statements so you can see exactly what happens under the hood as you learn MCP.

## 1. Imports and Setup

First, we import the necessary libraries and load our environment variables (importantly, `OPENAI_API_KEY`).

In [1]:
import asyncio
import json
import subprocess
from contextlib import AsyncExitStack
from typing import Any, Dict, List, Optional

import nest_asyncio
from dotenv import load_dotenv
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from openai import AsyncOpenAI

# Apply nest_asyncio to allow nested event loops in Jupyter
nest_asyncio.apply()

# Load environment variables (Make sure your .env has OPENAI_API_KEY)
load_dotenv("../.env", override=True)
print("Environment variables loaded.")

Environment variables loaded.


## 2. Defining the Class and Initialization

We start by defining the `MCPOpenAIClient` class and its `__init__` method. This sets up the instance variables we need to keep track of the session.

In [2]:
class MCPOpenAIClient:
    """Client for interacting with OpenAI models using MCP tools."""

    def __init__(self, model: str = "gpt-4o-mini"):
        """Initialize the OpenAI MCP client."""
        print(f"Initializing MCPOpenAIClient with model: {model}")
        
        # Initialize session and client objects
        self.session: Optional[ClientSession] = None
        self.exit_stack = AsyncExitStack()
        self.openai_client = AsyncOpenAI()
        self.model = model
        self.stdio: Optional[Any] = None
        self.write: Optional[Any] = None
        
        print("Initialization complete.")

## 3. Connecting to the Server

Next, we define the `connect_to_server` method and attach it to our class. This establishes the `stdio` transport connection to the `server.py` script.

In [3]:
async def connect_to_server(self, server_script_path: str = "server.py"):
    """Connect to an MCP server."""
    print(f"Connecting to MCP server using script: {server_script_path}...")
    
    # Server configuration
    server_params = StdioServerParameters(
        command="python",
        args=[server_script_path],
    )

    # Connect to the server
    stdio_transport = await self.exit_stack.enter_async_context(
        stdio_client(server_params, errlog=subprocess.DEVNULL)
    )
    self.stdio, self.write = stdio_transport
    self.session = await self.exit_stack.enter_async_context(
        ClientSession(self.stdio, self.write)
    )

    # Initialize the connection
    await self.session.initialize()
    print("Server connection initialized!")

    # List available tools
    tools_result = await self.session.list_tools()
    print("\nConnected to server with tools:")
    for tool in tools_result.tools:
        print(f"  - {tool.name}: {tool.description}")

# Attach method to the class dynamically
MCPOpenAIClient.connect_to_server = connect_to_server

## 4. Formatting Tools for OpenAI

OpenAI requires tool descriptions in a specific JSON schema. This method retrieves the tools from the MCP server and formats them accordingly.

In [4]:
async def get_mcp_tools(self) -> List[Dict[str, Any]]:
    """Get available tools from the MCP server in OpenAI format."""
    print("Fetching tools from MCP server to pass to OpenAI...")
    
    tools_result = await self.session.list_tools()
    tools_list = [
        {
            "type": "function",
            "function": {
                "name": tool.name,
                "description": tool.description,
                "parameters": tool.inputSchema,
            },
        }
        for tool in tools_result.tools
    ]
    
    print(f"Formatted {len(tools_list)} tool(s) for OpenAI format.")
    return tools_list

# Attach method to the class dynamically
MCPOpenAIClient.get_mcp_tools = get_mcp_tools

## 5. Processing the Query

This method contains the core logic: it sends a user query to OpenAI, executes tools if the LLM requests them, and finally requests a complete natural language response.

In [5]:
async def process_query(self, query: str) -> str:
    """Process a query using OpenAI and available MCP tools."""
    print(f"\n--- Processing query: '{query}' ---")

    # Get available tools formatted for OpenAI
    tools = await self.get_mcp_tools()

    # 1. Initial OpenAI API call
    print("\n1. Sending initial request to OpenAI with tool list...")
    response = await self.openai_client.chat.completions.create(
        model=self.model,
        messages=[{"role": "user", "content": query}],
        tools=tools,
        tool_choice="auto",
    )

    # Get assistant's response
    assistant_message = response.choices[0].message

    # Initialize conversation history with user query and assistant response
    messages = [
        {"role": "user", "content": query},
        assistant_message,
    ]

    # Handle tool calls if present
    if assistant_message.tool_calls:
        print(f"\n2. OpenAI decided to call {len(assistant_message.tool_calls)} tool(s)!")
        
        # Process each tool call
        for tool_call in assistant_message.tool_calls:
            print(f"  -> Executing tool: {tool_call.function.name}")
            print(f"  -> With arguments: {tool_call.function.arguments}")
            
            # Execute tool call on the MCP Server
            result = await self.session.call_tool(
                tool_call.function.name,
                arguments=json.loads(tool_call.function.arguments),
            )
            
            print(f"  -> Tool execution complete. Result returned {len(result.content[0].text)} characters.")

            # Add tool response to conversation history
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result.content[0].text,
                }
            )

        # 3. Get final response from OpenAI with tool results
        print("\n3. Sending tool results back to OpenAI for a final answer...")
        final_response = await self.openai_client.chat.completions.create(
            model=self.model,
            messages=messages,
            tools=tools,
            tool_choice="none",  # Don't allow more tool calls
        )
        print("  -> Received final answer from OpenAI.")
        return final_response.choices[0].message.content

    # No tool calls, just return the direct response
    print("\n2. No tool calls were needed. Returning direct response.")
    return assistant_message.content

# Attach method to the class dynamically
MCPOpenAIClient.process_query = process_query

## 6. Cleanup Method

To properly close our async connections, we create a `cleanup` method.

In [6]:
async def cleanup(self):
    """Clean up resources."""
    print("\nCleaning up resources and closing connection...")
    await self.exit_stack.aclose()
    print("Cleanup complete.")

# Attach method to the class dynamically
MCPOpenAIClient.cleanup = cleanup

## 7. Execution Time!

Now that the `MCPOpenAIClient` class is fully assembled, we can instantiate it, connect to the server, ask a question, and finally clean up the resources.

In [7]:
# 1. Instantiate the client
client = MCPOpenAIClient(model="gpt-4o-mini")

# 2. Connect to the server
await client.connect_to_server("server.py")

# 3. Execute a query
query = "What is our company's vacation policy?"
response = await client.process_query(query)

# 4. Print the final response
print("\n" + "="*50)
print("FINAL RESPONSE:\n")
print(response)
print("="*50)

# 5. Clean up
await client.cleanup()

Initializing MCPOpenAIClient with model: gpt-4o-mini
Initialization complete.
Connecting to MCP server using script: server.py...
Server connection initialized!

Connected to server with tools:
  - get_knowledge_base: Retrieve the entire knowledge base as a formatted string.

    Returns:
        A formatted string containing all Q&A pairs from the knowledge base.
    

--- Processing query: 'What is our company's vacation policy?' ---
Fetching tools from MCP server to pass to OpenAI...
Formatted 1 tool(s) for OpenAI format.

1. Sending initial request to OpenAI with tool list...

2. OpenAI decided to call 1 tool(s)!
  -> Executing tool: get_knowledge_base
  -> With arguments: {}
  -> Tool execution complete. Result returned 1987 characters.

3. Sending tool results back to OpenAI for a final answer...
  -> Received final answer from OpenAI.

FINAL RESPONSE:

Our company's vacation policy states that full-time employees are entitled to 20 paid vacation days per year, which can be taken